In [1]:
%pip install -q numpy pandas torch transformers sentence-transformers gensim openai accelerate ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import numpy as np
import pandas as pd
import torch
import transformers

In [3]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")

print("vocabulary size:", len(glove.key_to_index))
print("vector size    :", glove.vector_size)
print("vector of 'server' (first 6 numbers):", glove["server"][:6].round(2))

vocabulary size: 400000
vector size    : 50
vector of 'server' (first 6 numbers): [ 0.77 -0.38  1.12  1.77  0.08 -0.89]


In [4]:
print("15 nearest to 'bank':", [w for w, _ in glove.most_similar("bank", topn=15)])
print()
for word in ["loan", "money", "river", "water"]:
    print(f"bank vs {word:6s} cosine = {glove.similarity('bank', word):.2f}")

15 nearest to 'bank': ['banks', 'securities', 'banking', 'investment', 'exchange', 'financial', 'credit', 'lender', 'capital', 'brokerage', 'currency', 'investments', 'funds', 'fund', 'assets']

bank vs loan   cosine = 0.68
bank vs money  cosine = 0.64
bank vs river  cosine = 0.41
bank vs water  cosine = 0.34


In [5]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
def glove_sentence_vector(sentence):
    return np.mean([glove[word] for word in sentence.lower().split()], axis=0)

a = glove_sentence_vector("dog bites man")
b = glove_sentence_vector("man bites dog")
print("'dog bites man' vs 'man bites dog': cosine =", round(cosine(a, b), 2))

'dog bites man' vs 'man bites dog': cosine = 1.0


In [6]:
from transformers import AutoTokenizer, AutoModel

bert_name = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name)
bert_model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [7]:
def bank_contextual(text):
    enc = bert_tokenizer(text, return_tensors="pt")
    with torch.no_grad():                                        # prediction only, no gradients needed
        hidden = bert_model(**enc).last_hidden_state[0]          # one vector per token, after the layers
    position = bert_tokenizer.convert_ids_to_tokens(enc["input_ids"][0]).index("bank")
    return hidden[position].numpy()

river1 = bank_contextual("the fisherman sat on the bank of the river")
river2 = bank_contextual("we walked along the muddy river bank")
money = bank_contextual("the bank approved my home loan")

print("GloVe (static)                  : 'bank' vs 'bank' = 1.00 (always the same vector)")
print("transformer: river  vs river2  :", round(cosine(river1, river2), 2))
print("transformer: river bank 1 vs money bank:", round(cosine(river1, money), 2))
print("transformer: river bank 2 vs money bank:", round(cosine(river2, money), 2))

GloVe (static)                  : 'bank' vs 'bank' = 1.00 (always the same vector)
transformer: river  vs river2  : 0.77
transformer: river bank 1 vs money bank: 0.58
transformer: river bank 2 vs money bank: 0.55


In [8]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sentences = {
    "A1": "pods keep restarting after the release",
    "A2": "container crash loop following the deployment",
    "B1": "disk is almost full on the database server",
    "B2": "no storage space left on the DB host",
    "C":  "my favourite pizza has extra cheese",
}
vectors = st_model.encode(list(sentences.values()), normalize_embeddings=True)      # (5, 384), every row has length 1
similarity = vectors @ vectors.T                                                     # dot product of unit vectors = cosine

print(pd.DataFrame(similarity, index=sentences.keys(), columns=sentences.keys()).round(2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

      A1    A2    B1    B2     C
A1  1.00  0.43  0.13  0.15 -0.10
A2  0.43  1.00  0.12  0.23 -0.06
B1  0.13  0.12  1.00  0.69  0.05
B2  0.15  0.23  0.69  1.00  0.02
C  -0.10 -0.06  0.05  0.02  1.00


In [9]:
runbooks = [
    "Fix CrashLoopBackOff: pod keeps restarting",
    "Disk full: clean up logs and expand the volume",
    "High CPU on a node: find the noisy process",
    "Certificate expired: renew the TLS certificate",
    "Database connection pool exhausted",
]
runbook_vectors = st_model.encode(runbooks, normalize_embeddings=True)

alert = "containers restart over and over after the new image was deployed"
scores = runbook_vectors @ st_model.encode(alert, normalize_embeddings=True)
for i in scores.argsort()[::-1]:
    print(f"{scores[i]:.2f}  {runbooks[i]}")

0.40  Fix CrashLoopBackOff: pod keeps restarting
0.22  Database connection pool exhausted
0.20  Certificate expired: renew the TLS certificate
0.10  Disk full: clean up logs and expand the volume
0.08  High CPU on a node: find the noisy process


In [10]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [11]:
tok = GPT2Tokenizer.from_pretrained("gpt2")

In [12]:
model = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="eager") 

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [13]:
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [14]:
text = "Rose loves sour fresh fruits like strawberries, apples"

In [15]:
inputs = tok(text, return_tensors="pt")

In [16]:
inputs

{'input_ids': tensor([[31087, 10408, 11348,  4713, 15921,   588, 36973,    11, 22514]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [17]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [18]:
logits

tensor([[[ -31.2852,  -30.6273,  -34.2245,  ...,  -39.2340,  -38.1299,
           -31.6271],
         [ -74.0125,  -75.1880,  -81.0360,  ...,  -81.6594,  -82.0551,
           -76.6324],
         [ -81.2202,  -82.9256,  -88.6227,  ...,  -90.3833,  -91.0548,
           -84.3656],
         ...,
         [ -61.7508,  -65.3426,  -70.5698,  ...,  -77.2291,  -74.8082,
           -66.6509],
         [ -98.4999, -101.2800, -103.2987,  ..., -107.5981, -103.5498,
          -100.3659],
         [ -21.2307,  -23.4124,  -27.5964,  ...,  -34.3731,  -31.4195,
           -25.1886]]])

In [19]:
print("logits shape:", logits.shape, "= (sentences, tokens, vocabulary size)")

logits shape: torch.Size([1, 9, 50257]) = (sentences, tokens, vocabulary size)


In [20]:
next_token_logits = logits[:, -1, :]

In [21]:
probs = torch.softmax(next_token_logits, dim=-1) 

In [22]:
probs

tensor([[9.0013e-05, 1.0158e-05, 1.5478e-07,  ..., 1.7646e-10, 3.3837e-09,
         1.7195e-06]])

In [23]:
top_k = torch.topk(probs, k=5)  

In [24]:
top_k

torch.return_types.topk(
values=tensor([[0.6939, 0.2731, 0.0073, 0.0054, 0.0048]]),
indices=tensor([[  11,  290,  559,  393, 1222]]))

In [25]:
for idx, score in zip(top_k.indices[0], top_k.values[0]):
    predicted_word = tok.decode([idx])
    print(f"{predicted_word!r} : {float(score):.4f}")

',' : 0.6939
' and' : 0.2731
'au' : 0.0073
' or' : 0.0054
' &' : 0.0048


In [32]:
text = "The kids are watching this movie with their parents and they want to watch the movie and see what happens. They  want  to  see what"
inputs = tok(text, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits

probs = torch.softmax(logits[:, -1, :], dim=-1)
top_k = torch.topk(probs, k=5)
for idx, score in zip(top_k.indices[0], top_k.values[0]):
    predicted_word = tok.decode([idx])
    print(f"{predicted_word!r} : {float(score):.4f}")

' what' : 0.3453
' the' : 0.1444
' ' : 0.0789
' how' : 0.0504
' it' : 0.0471


In [36]:
from transformers import AutoModelForSeq2SeqLM

t5_name = "google/flan-t5-base"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name).to("cuda" if torch.cuda.is_available() else "cpu")

prompt = "Can you write a short beautiful love story?"
inputs = t5_tokenizer(prompt, return_tensors="pt").to(t5_model.device)

output = t5_model.generate(**inputs, max_new_tokens=100, no_repeat_ngram_size=3)      # stops it repeating the same phrase
print(t5_tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


The story of a young man who has been a sailor for a long time. He is a very talented stag, who is very good at spotting the elusive elusive fish. He has a great sense of humor and is very lovable. He loves to go fishing and is able to get away from his stumbling blocks. He also loves to be around his family and friends. He likes to go to the


In [38]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

t5_name = "google/flan-t5-base"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

prompt = "Can you write a short beautiful love story?"
inputs = t5_tokenizer(prompt, return_tensors="pt").to(t5_model.device)

output = t5_model.generate(
    **inputs,
    max_new_tokens=100,
    no_repeat_ngram_size=3,
    do_sample=True,
    temperature=0.7
)

print(t5_tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


One night during a party, a man suddenly runs into a woman. She pulls her hand out of her purse and runs to her. She runs back to her car, quickly finding the man outside. She drives home and she sees the man walking by. She quickly runs out of the house and runs into the woman next to her, who is her mother.


In [40]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

t5_name = "google/flan-t5-base"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

prompt = "Can you write a short beautiful love story?"
inputs = t5_tokenizer(prompt, return_tensors="pt").to(t5_model.device)

output = t5_model.generate(
    **inputs,
    max_new_tokens=100,
    no_repeat_ngram_size=3,
    do_sample=True,
    temperature=0.7,
    top_k=50
)

print(t5_tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


The first time I met a woman I thought was an angel. She was very pretty and very handsome. I had a lovely time together. I will never forget her. I always thought that she was very sexy and frothy. I'd say that's true. I like to be alone and I'm not very sociable, so I just kept myself busy. But then I've got my first crush, a man who I'
